# Missing values

A hole in the table is the most common defect in real data. This notebook walks the classical
toolkit for handling it, shows how much of that toolkit TabPFN makes unnecessary, and isolates the
two places where it still matters. Every experiment is a small, self-contained synthetic
construction, so the behaviour we observe is real but the setup stays easy to read.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from tabpfn import TabPFNClassifier
from tqdm import tqdm
from tabulate import tabulate

def printdf(df, nrows=5, showindex=True): print(tabulate(df.head(nrows), headers='keys', tablefmt='psql', showindex=showindex))

def auc(model, X_train, y_train, X_test, y_test):
    """Fit a model and return its test ROC AUC, rounded."""
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    return round(roc_auc_score(y_test, probabilities), 4)


def relative_to(table, baseline_col):
    """Each column minus a baseline column: + helps, - hurts versus that reference."""
    return table.sub(table[baseline_col], axis=0).round(4)

from sklearn.ensemble import RandomForestClassifier

# One shared roster, used by every experiment below.
MODELS = {
    "TabPFN":       lambda: TabPFNClassifier(random_state=0),
    "LightGBM":     lambda: LGBMClassifier(verbose=-1),
    "XGBoost":      lambda: XGBClassifier(tree_method="hist", eval_metric="logloss"),
    "CatBoost":     lambda: CatBoostClassifier(verbose=0),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=200, random_state=0),
    "Linear":       lambda: Pipeline([("impute", SimpleImputer()), ("scale", StandardScaler()),
                                      ("logreg", LogisticRegression(max_iter=2000))]),
}

## Classical toolkit

The classical toolkit, in rough order of effort:

- **Drop** rows or columns when the gaps are few and uninformative.
- **Fill** with the median or mode, or a group-wise statistic (learned on train only).
- **Flag** with a missing-indicator column, because absence can be predictive (especially MNAR).
- **Sentinel-encode** (`-9999`) so a tree can split missing into its own branch.
- **Model-based imputation** (MICE, KNN) to reconstruct a column from the others.
- **For tree models, do not impute**: XGBoost, LightGBM and CatBoost take NaNs natively and learn
  where to route them.

Which one you reach for depends on the model, and on *why* the value is missing (MCAR, MAR, MNAR).

## Missingness mechanisms

Before choosing a fix, the textbook asks *why* a value is missing, because that decides whether the
gap is noise or signal:

- **MCAR (Missing Completely At Random).** No pattern at all, e.g. a sensor that randomly drops
  readings. That a value is missing tells you nothing.
- **MAR (Missing At Random).** The gap depends on *other observed* columns, e.g. older customers
  skip the income field. You can predict the missingness from what you do observe, so it is
  recoverable.
- **MNAR (Missing Not At Random).** The gap depends on the *missing value itself*, e.g. high
  earners decline to report income. The missingness carries information you cannot fully recover
  from the other columns.

The upshot: under MAR and MNAR, "this value is missing" is itself a feature. That is what a
missing-indicator captures, and why imputing-and-forgetting can throw signal away.

In [2]:
# Three ways a column could go missing. We measure how much "this value is missing"
# correlates with the label under each.
rng = np.random.RandomState(0)
n = 6000

x = rng.randn(n)
other = rng.randn(n)
label = (rng.rand(n) < 1 / (1 + np.exp(-(x + other)))).astype(int)   # label depends on x and other

is_missing = {
    "MCAR (random)":            rng.rand(n) < 0.3,   # no pattern
    "MAR  (depends on other)":  other > 0.5,         # depends on an observed column
    "MNAR (depends on x)":      x > 0.5,             # depends on the missing value itself
}

correlations = {}
for name, mask in is_missing.items():
    correlations[name] = round(abs(np.corrcoef(mask.astype(int), label)[0, 1]), 3)

printdf(pd.DataFrame(correlations.items(), columns=["missing_type", "corr(is_missing, label)"]))

+----+-------------------------+---------------------------+
|    | missing_type            |   corr(is_missing, label) |
|----+-------------------------+---------------------------|
|  0 | MCAR (random)           |                     0.014 |
|  1 | MAR  (depends on other) |                     0.266 |
|  2 | MNAR (depends on x)     |                     0.267 |
+----+-------------------------+---------------------------+


MCAR's indicator is uninformative (~0); under MAR and MNAR the missing-flag correlates with the
label. The difference between those two: MAR's signal is recoverable from `other` (a column you
still observe), while MNAR's lives in the value you can no longer see. That distinction is exactly
where an indicator earns or wastes its place.

## Native handling

Leave the NaNs in. Inside the model, before the forward pass, every missing cell is (1) filled with
that feature's **mean over the training rows**, and (2) recorded in a **per-feature
missing-indicator channel** that the network reads. So TabPFN performs *mean-impute-plus-indicator*
by default, the move strong neural baselines do by hand. Most of the checklist above is already
inside the model. The rest of this notebook asks: which parts does that make redundant, and where
does the toolkit still earn its keep?

In [3]:
# TabPFN accepts NaNs directly. No imputation step needed.
demo = pd.DataFrame(np.random.RandomState(0).randn(200, 3), columns=["a", "b", "c"])
demo.loc[:60, "a"] = np.nan                  # put some holes in column 'a'
demo_label = (demo["b"] > 0).astype(int)

model = TabPFNClassifier(random_state=0)
model.fit(demo, demo_label)                  # fits fine, NaNs and all
model.predict_proba(demo)[:5, 1]             # probabilities for the first five rows

array([9.9998701e-01, 9.9998701e-01, 1.5486538e-04, 9.9990189e-01,
       9.9980223e-01], dtype=float32)

### Can you change how TabPFN handles missing values?

Short answer: no. The mean-fill and the indicator channel are baked into the v3 architecture and
its pretrained weights, not exposed as settings. No constructor argument and no `inference_config`
field controls feature imputation. (The two parameters that touch missing values do so only
indirectly: `categorical_features_indices`, which decides whether a hole-bearing column is encoded
as categorical or numeric, and `inference_config`, which sets the scaling the filled value then
passes through.)

In [4]:
import inspect

# 1) The constructor: no argument is about missing values or imputation.
constructor_params = [p for p in inspect.signature(TabPFNClassifier.__init__).parameters if p != "self"]
print("constructor parameters:")
print(" ", constructor_params)

# 2) Confirm the loaded model is v3, with the indicator channel on.
fitted = TabPFNClassifier(random_state=0).fit(demo, demo_label)
print("\nloaded model:", type(fitted.model_).__name__,
      "| use_nan_indicators =", fitted.model_.use_nan_indicators)

# 3) The preprocessing knobs (inference_config): none controls feature imputation.
config_fields = [f for f in dir(fitted.inference_config_) if f.isupper()]
imputation_knobs = [f for f in config_fields if any(k in f for k in ("IMPUT", "FILL"))]
print("\ninference_config fields that control feature imputation:", imputation_knobs or "none")

constructor parameters:
  ['n_estimators', 'categorical_features_indices', 'softmax_temperature', 'balance_probabilities', 'average_before_softmax', 'model_path', 'device', 'ignore_pretraining_limits', 'inference_precision', 'fit_mode', 'memory_saving_mode', 'random_state', 'n_jobs', 'n_preprocessing_jobs', 'inference_config', 'differentiable_input', 'eval_metric', 'tuning_config', 'show_progress_bar']



loaded model: TabPFNV3 | use_nan_indicators = True

inference_config fields that control feature imputation: none


## Classical fills

A useful numeric column `a` is 40% missing (completely at random), and it varies by an observed
segment `seg` (think product or region). We try the classical fills, mean, median, an out-of-range
sentinel, a segment-wise mean, and mean-plus-indicator, against just leaving the NaN in. (Mode is
the same idea for a categorical column: fill with the most frequent level.) How much does the
choice move each model?

In [5]:
# Build a small dataset: 'a' depends on a segment/group, the label depends on 'a' and 'b'.
rng = np.random.RandomState(0)
n = 6000

seg = rng.randint(0, 4, n)                          # segment 0-3 (e.g. region), observed
segment_value = np.array([4.0, -2.0, 6.0, -1.0])    # each segment's typical 'a' (jumps around)
a = segment_value[seg] + rng.randn(n)               # 'a' = its segment's value + noise
b = rng.randn(n)                                    # a second predictor

score = 0.6 * a + b
label = (rng.rand(n) < 1 / (1 + np.exp(-score))).astype(int)   # 0/1 label from a sigmoid of score

data = pd.DataFrame({"a": a, "b": b, "seg": seg.astype(float)})
data.loc[rng.rand(n) < 0.40, "a"] = np.nan          # knock out 40% of 'a', at random

train, test = data.iloc[:n // 2], data.iloc[n // 2:]
y_train, y_test = label[:n // 2], label[n // 2:]

In [6]:
# Fill values, all computed on the TRAIN half only (no leakage).
train_statistics = {
    "mean": train["a"].mean(),
    "median": train["a"].median(),
    "segment_mean": train.groupby("seg")["a"].mean(),   # mean of 'a' within each segment
}

def filled(frame, method, train_statistics):
    """Return a copy of `frame` with column 'a' handled by the chosen method.

    The fill values arrive through `train_statistics` (all computed on the training
    half), so the dependence on train-only quantities is explicit in the signature
    rather than hidden in globals. That is the discipline that keeps the fill leakage-free.
    """
    out = frame.copy()
    if method == "mean":
        out["a"] = out["a"].fillna(train_statistics["mean"])
    elif method == "median":
        out["a"] = out["a"].fillna(train_statistics["median"])
    elif method == "sentinel":
        out["a"] = out["a"].fillna(-9999)
    elif method == "group":
        out["a"] = out["a"].fillna(out["seg"].map(train_statistics["segment_mean"]))
    elif method == "indicator":
        out["a_missing"] = out["a"].isna().astype(int)
        out["a"] = out["a"].fillna(train_statistics["mean"])
    # "native" is not handled, so the NaN is left in place
    return out

In [7]:
methods = ["native", "mean", "median", "sentinel", "group", "indicator"]

scores = {}
for model_name, make_model in MODELS.items():
    row = {}
    for method in tqdm(methods, desc=model_name, leave=False):
        train_filled = filled(train, method, train_statistics)
        test_filled = filled(test, method, train_statistics)
        row[method] = auc(make_model(), train_filled, y_train, test_filled, y_test)
    scores[model_name] = row

fills_table = pd.DataFrame(scores).T
printdf(fills_table, nrows=len(MODELS))
print("\nSame table, relative to leaving the NaN in place (native), + helps / - hurts:")
printdf(relative_to(fills_table, "native"), nrows=len(MODELS))

TabPFN:   0%|                                                       | 0/6 [00:00<?, ?it/s]

TabPFN:  17%|███████▊                                       | 1/6 [00:02<00:12,  2.45s/it]

TabPFN:  33%|███████████████▋                               | 2/6 [00:04<00:09,  2.35s/it]

TabPFN:  50%|███████████████████████▌                       | 3/6 [00:07<00:07,  2.43s/it]

TabPFN:  67%|███████████████████████████████▎               | 4/6 [00:09<00:04,  2.38s/it]

...

RandomForest:  83%|██████████████████████████████████▏      | 5/6 [00:03<00:00,  1.40it/s]

RandomForest: 100%|█████████████████████████████████████████| 6/6 [00:04<00:00,  1.36it/s]

Linear:   0%|                                                       | 0/6 [00:00<?, ?it/s]

Linear: 100%|███████████████████████████████████████████████| 6/6 [00:00<00:00, 23.00it/s]

+--------------+----------+--------+----------+------------+---------+-------------+
|              |   native |   mean |   median |   sentinel |   group |   indicator |
|--------------+----------+--------+----------+------------+---------+-------------|
| TabPFN       |   0.8918 | 0.8907 |   0.8906 |     0.8887 |  0.8915 |      0.8914 |
| LightGBM     |   0.872  | 0.8713 |   0.8716 |     0.8719 |  0.8706 |      0.8695 |
| XGBoost      |   0.8567 | 0.8591 |   0.8597 |     0.8608 |  0.8594 |      0.8596 |
| CatBoost     |   0.8825 | 0.8807 |   0.8809 |     0.8838 |  0.8838 |      0.8821 |
| RandomForest |   0.8484 | 0.8484 |   0.8483 |     0.8479 |  0.8493 |      0.8459 |
| Linear       |   0.8367 | 0.8367 |   0.8372 |     0.6682 |  0.8922 |      0.8365 |
+--------------+----------+--------+----------+------------+---------+-------------+

Same table, relative to leaving the NaN in place (native), + helps / - hurts:
+--------------+----------+---------+----------+------------+---------+

Read across each row. **The native-handling models barely move**: they read the segment directly, so how
you fill `a` hardly matters. The **linear model is the sensitive one**: the segment-wise `group`
fill helps it (it cannot use `seg` non-monotonically on its own), and the `-9999` sentinel hurts it
(a giant value wrecks a linear fit, while a tree just splits it off). The `indicator` adds nothing
here, because this gap is MCAR, there is no missingness signal to capture. That is the grounding in
one table: for a flexible model the fill is mostly a non-decision; the care goes into weak models
and into *informative* missingness, which is where the rest of the notebook goes.

To check that group-fill's benefit is entirely about *access to the grouping*, run the same
comparison but drop `seg` from the columns the models see. The fill still uses `seg` to compute the
group mean; the models just no longer get `seg` as a feature, so the only way the segment can reach
them is through the `group` fill.

In [8]:
# Same comparison, but 'seg' is removed from the features (group-fill still uses it to fill 'a').
scores_no_seg = {}
for model_name, make_model in MODELS.items():
    row = {}
    for method in tqdm(methods, desc=model_name, leave=False):
        train_f = filled(train, method, train_statistics).drop(columns=["seg"])
        test_f = filled(test, method, train_statistics).drop(columns=["seg"])
        row[method] = auc(make_model(), train_f, y_train, test_f, y_test)
    scores_no_seg[model_name] = row

fills_no_seg = pd.DataFrame(scores_no_seg).T
printdf(fills_no_seg, nrows=len(MODELS))
print("\nSame table, relative to native (now with the seg column removed):")
printdf(relative_to(fills_no_seg, "native"), nrows=len(MODELS))

TabPFN:   0%|                                                       | 0/6 [00:00<?, ?it/s]

TabPFN:  17%|███████▊                                       | 1/6 [00:02<00:11,  2.31s/it]

TabPFN:  33%|███████████████▋                               | 2/6 [00:04<00:09,  2.29s/it]

TabPFN:  50%|███████████████████████▌                       | 3/6 [00:06<00:06,  2.27s/it]

TabPFN:  67%|███████████████████████████████▎               | 4/6 [00:09<00:04,  2.26s/it]

...

RandomForest:  67%|███████████████████████████▎             | 4/6 [00:03<00:01,  1.22it/s]

RandomForest:  83%|██████████████████████████████████▏      | 5/6 [00:04<00:00,  1.26it/s]

RandomForest: 100%|█████████████████████████████████████████| 6/6 [00:04<00:00,  1.25it/s]

Linear:   0%|                                                       | 0/6 [00:00<?, ?it/s]

+--------------+----------+--------+----------+------------+---------+-------------+
|              |   native |   mean |   median |   sentinel |   group |   indicator |
|--------------+----------+--------+----------+------------+---------+-------------|
| TabPFN       |   0.8328 | 0.8313 |   0.8314 |     0.8124 |  0.8917 |      0.8316 |
| LightGBM     |   0.8051 | 0.8063 |   0.8059 |     0.8036 |  0.8714 |      0.8049 |
| XGBoost      |   0.794  | 0.7936 |   0.7927 |     0.7929 |  0.861  |      0.7948 |
| CatBoost     |   0.8185 | 0.8211 |   0.8192 |     0.8199 |  0.8836 |      0.8211 |
| RandomForest |   0.7612 | 0.7598 |   0.7607 |     0.7579 |  0.8511 |      0.7581 |
| Linear       |   0.8315 | 0.8315 |   0.8322 |     0.6257 |  0.8921 |      0.8315 |
+--------------+----------+--------+----------+------------+---------+-------------+

Same table, relative to native (now with the seg column removed):
+--------------+----------+---------+----------+------------+---------+------------

Now `group` jumps for **every** model (each gains roughly +0.06 to +0.09), while the
other fills stay flat. With no `seg` column, group-fill is the only channel for the segment, so it
is the only fill that carries information. The takeaway, stated once: **group-fill helps exactly
when the model cannot already see the grouping.** A flexible model that has the segment column gets
nothing from group-filling; strip the column and it becomes the best fill there is.

## The toolkit mostly collapses

A classic reason to impute carefully is that a missing column can be *reconstructed* from the
others (model-based imputation). The next few cells build exactly that, with a few small
helpers:

- `make_data(make_Z)` builds a table of random columns `x0..x9` plus a column `Z` that is a chosen
  function of them, with the label depending on `Z`. So `Z` is useful, and reconstructable.
- `hide_half_of_Z(...)` knocks out half of `Z` at random.
- `model_based_impute(...)` trains a small regressor to predict `Z` from the other columns and fills the
  holes with its predictions.
- `compare_imputations(...)` scores four ways of handling the holes (the true `Z`, leaving the NaN,
  mean-fill, and model-based) across the three models. (MICE is the iterative, multi-column version of this; KNN is another model-based method.)

In [9]:
def make_data(make_Z, n=8000, seed=0):
    """Build a table where column Z is a function of x0..x9, and the label depends on Z."""
    rng = np.random.RandomState(seed)
    X = rng.randn(n, 10)

    signal = make_Z(X)                              # Z's underlying signal, built from x0..x9
    signal = (signal - signal.mean()) / signal.std()
    Z = signal + 0.5 * rng.randn(n)                 # the actual column: signal plus noise

    label = (rng.rand(n) < 1 / (1 + np.exp(-2.5 * signal))).astype(int)

    columns = [f"x{i}" for i in range(10)] + ["Z"]
    table = pd.DataFrame(np.column_stack([X, Z]), columns=columns)
    return table, label

def hide_half_of_Z(table, seed):
    """Set half of column Z to NaN, at random."""
    table = table.copy()
    holes = np.random.RandomState(seed).rand(len(table)) < 0.5
    table.loc[holes, "Z"] = np.nan
    return table

def model_based_impute(train, test):
    """Model-based imputation: predict Z from the other columns, fill the holes with the prediction."""
    other_columns = [c for c in train.columns if c != "Z"]
    observed = train["Z"].notna()

    regressor = LGBMRegressor(n_estimators=400, verbose=-1)
    regressor.fit(train.loc[observed, other_columns], train.loc[observed, "Z"])

    train, test = train.copy(), test.copy()
    for frame in (train, test):
        missing = frame["Z"].isna()
        frame.loc[missing, "Z"] = regressor.predict(frame.loc[missing, other_columns])
    return train, test

In [10]:
def compare_imputations(table, label):
    """Hide half of Z, then score the handling options across the full model roster."""
    half = len(table) // 2
    train, test = table.iloc[:half], table.iloc[half:]
    y_train, y_test = label[:half], label[half:]

    train_holes = hide_half_of_Z(train, seed=1)
    test_holes = hide_half_of_Z(test, seed=2)
    column_means = train_holes.mean()

    variants = {
        "oracle (true Z)": (train, test),
        "native (NaN)":    (train_holes, test_holes),
        "mean-impute":     (train_holes.fillna(column_means), test_holes.fillna(column_means)),
        "model-based":     model_based_impute(train_holes, test_holes),
    }

    scores = {}
    for model_name, make_model in MODELS.items():
        row = {}
        for variant_name, (train_variant, test_variant) in tqdm(variants.items(), desc=model_name, leave=False):
            row[variant_name] = auc(make_model(), train_variant, y_train, test_variant, y_test)
        scores[model_name] = row
    return pd.DataFrame(scores).T

def relative_to_oracle(scores, oracle_col="oracle (true Z)"):
  """Each option as its % change in AUC vs the oracle (true Z). Negative = below the ceiling."""
  rel = scores.sub(scores[oracle_col], axis=0).div(scores[oracle_col], axis=0).mul(100)
  rel = rel.drop(columns=[oracle_col])
  rel.columns = [c + " Δ% vs oracle" for c in rel.columns]
  return rel.round(2)

In [11]:
# An easy Z: a small combination of three columns.
def easy_Z(X):
    return X[:, 0] * X[:, 1] + X[:, 2] ** 2

table, label = make_data(easy_Z)
scores = compare_imputations(table, label)
printdf(scores, nrows=len(MODELS))
printdf(relative_to_oracle(scores), nrows=len(MODELS)) 

TabPFN:   0%|                                                       | 0/4 [00:00<?, ?it/s]

TabPFN:  25%|███████████▊                                   | 1/4 [00:03<00:10,  3.49s/it]

TabPFN:  50%|███████████████████████▌                       | 2/4 [00:06<00:06,  3.48s/it]

TabPFN:  75%|███████████████████████████████████▎           | 3/4 [00:10<00:03,  3.47s/it]

TabPFN: 100%|███████████████████████████████████████████████| 4/4 [00:13<00:00,  3.48s/it]

...

Linear:  25%|███████████▊                                   | 1/4 [00:00<00:00,  7.71it/s]

Linear:  50%|███████████████████████▌                       | 2/4 [00:00<00:00,  4.56it/s]

Linear:  75%|███████████████████████████████████▎           | 3/4 [00:00<00:00,  3.24it/s]

Linear: 100%|███████████████████████████████████████████████| 4/4 [00:01<00:00,  3.80it/s]

+--------------+-------------------+----------------+---------------+---------------+
|              |   oracle (true Z) |   native (NaN) |   mean-impute |   model-based |
|--------------+-------------------+----------------+---------------+---------------|
| TabPFN       |            0.8326 |         0.8313 |        0.831  |        0.8314 |
| LightGBM     |            0.8077 |         0.8047 |        0.8018 |        0.8049 |
| XGBoost      |            0.8012 |         0.7923 |        0.795  |        0.7991 |
| CatBoost     |            0.8205 |         0.8194 |        0.822  |        0.8187 |
| RandomForest |            0.807  |         0.7969 |        0.7989 |        0.8039 |
| Linear       |            0.7826 |         0.6559 |        0.6559 |        0.7903 |
+--------------+-------------------+----------------+---------------+---------------+
+--------------+-----------------------------+----------------------------+----------------------------+
|              |   native (NaN) Δ% 

For TabPFN and the tree models every option ties: leaving the hole, mean-filling, or carefully rebuilding
`Z` all land within noise, and all match the *oracle* that never lost `Z`. They reconstruct what
they need from the other columns internally. Rebuilding `Z` explicitly only rescues the **linear** model, which cannot
learn the combination itself. Model-based imputation is a crutch for weak models.

(We also checked hand-built indicators and `-9999` sentinels: for TabPFN both land within noise
too. It is forgiving about how you encode a hole.)

In [12]:
# A hard Z: a high-order, nonlinear combination of several columns.
def hard_Z(X):
    return (X[:, 0] * X[:, 1] * X[:, 2]
            + 1.5 * np.sin(3 * X[:, 3]) * np.cos(3 * X[:, 4])
            + 2.0 * ((X[:, 5] > 0) & (X[:, 6] > 0))
            - X[:, 7] ** 2)

table, label = make_data(hard_Z)
scores = compare_imputations(table, label)
printdf(scores, nrows=len(MODELS))
printdf(relative_to_oracle(scores), nrows=len(MODELS)) 

TabPFN:   0%|                                                       | 0/4 [00:00<?, ?it/s]

TabPFN:  25%|███████████▊                                   | 1/4 [00:03<00:10,  3.63s/it]

TabPFN:  50%|███████████████████████▌                       | 2/4 [00:07<00:07,  3.58s/it]

TabPFN:  75%|███████████████████████████████████▎           | 3/4 [00:10<00:03,  3.55s/it]

TabPFN: 100%|███████████████████████████████████████████████| 4/4 [00:14<00:00,  3.52s/it]

...

Linear:  25%|███████████▊                                   | 1/4 [00:00<00:00,  7.96it/s]

Linear:  50%|███████████████████████▌                       | 2/4 [00:00<00:00,  4.87it/s]

Linear:  75%|███████████████████████████████████▎           | 3/4 [00:00<00:00,  4.84it/s]

Linear: 100%|███████████████████████████████████████████████| 4/4 [00:00<00:00,  4.90it/s]

+--------------+-------------------+----------------+---------------+---------------+
|              |   oracle (true Z) |   native (NaN) |   mean-impute |   model-based |
|--------------+-------------------+----------------+---------------+---------------|
| TabPFN       |            0.8343 |         0.8002 |        0.7989 |        0.8155 |
| LightGBM     |            0.8208 |         0.7967 |        0.791  |        0.7999 |
| XGBoost      |            0.8118 |         0.7917 |        0.7906 |        0.7895 |
| CatBoost     |            0.8308 |         0.8186 |        0.8181 |        0.819  |
| RandomForest |            0.8283 |         0.7953 |        0.7965 |        0.8102 |
| Linear       |            0.8231 |         0.725  |        0.725  |        0.807  |
+--------------+-------------------+----------------+---------------+---------------+
+--------------+-----------------------------+----------------------------+----------------------------+
|              |   native (NaN) Δ% 

Now model-based imputation helps TabPFN and RandomForest as well, not just the linear model. And notice that `native`
TabPFN sits clearly below the `oracle`: even with every source column present, it cannot fully
rebuild a hard column in one forward pass. **TabPFN's internal reconstruction has a complexity
ceiling.** For a genuinely complex recoverable column, explicit model-based imputation still earns
its keep. That is the one real survivor of the classical toolkit.

## Failure modes

Native NaN handling has a catch that has nothing to do with TabPFN. A gradient-boosting model learns *where to
route a missing value* from the missingness it saw during training. Two ways that backfires, and
they land very differently for TabPFN.

### The vendor trap: a column missing only at inference

The classic schema or vendor-feed change: a feature was never missing in training, then a feed
drops in production. The model's routing for "missing" was never learned. Below, the test set
always has 30% of `x0` (a strong predictor) missing, and we sweep how much missingness the model
saw during training.

In [13]:
# A strong predictor x0; the label depends on it.
rng = np.random.RandomState(0)
X = rng.randn(8000, 10)
label = (rng.rand(8000) < 1 / (1 + np.exp(-(2.5 * X[:, 0] + X[:, 1])))).astype(int)
data = pd.DataFrame(X, columns=[f"x{i}" for i in range(10)])

half = 4000
train_full, test = data.iloc[:half], data.iloc[half:].copy()
y_train, y_test = label[:half], label[half:]

test.loc[rng.rand(half) < 0.30, "x0"] = np.nan      # TEST always has 30% of x0 missing

def train_with_missing(rate):
    """Copy the training set and make x0 missing at the given rate."""
    train = train_full.copy()
    holes = np.random.RandomState(1).rand(half) < rate
    train.loc[holes, "x0"] = np.nan
    return train

train_missing_rates = [0.0, 0.01, 0.05, 0.30]

scores = {}
for rate in train_missing_rates:
    column = f"{int(rate * 100)}% train-missing"
    train = train_with_missing(rate)
    scores[column] = {}
    for model_name, make_model in tqdm(MODELS.items(), desc=column, leave=False):
        scores[column][model_name] = auc(make_model(), train, y_train, test, y_test)

cliff_table = pd.DataFrame(scores)
printdf(cliff_table, nrows=len(MODELS))
print("\nSame table, relative to the 30% train-missing column (how far each model cliffs at 0%):")
printdf(relative_to(cliff_table, "30% train-missing"), nrows=len(MODELS))

0% train-missing:   0%|                                             | 0/6 [00:00<?, ?it/s]

0% train-missing:  17%|██████▏                              | 1/6 [00:03<00:17,  3.47s/it]

0% train-missing:  50%|██████████████████▌                  | 3/6 [00:03<00:02,  1.04it/s]

0% train-missing:  67%|████████████████████████▋            | 4/6 [00:05<00:02,  1.19s/it]

0% train-missing:  83%|██████████████████████████████▊      | 5/6 [00:07<00:01,  1.47s/it]

...

30% train-missing:  50%|██████████████████                  | 3/6 [00:03<00:02,  1.00it/s]

30% train-missing:  67%|████████████████████████            | 4/6 [00:05<00:02,  1.19s/it]

30% train-missing:  83%|██████████████████████████████      | 5/6 [00:07<00:01,  1.49s/it]

30% train-missing: 100%|████████████████████████████████████| 6/6 [00:07<00:00,  1.06s/it]

+--------------+--------------------+--------------------+--------------------+---------------------+
|              |   0% train-missing |   1% train-missing |   5% train-missing |   30% train-missing |
|--------------+--------------------+--------------------+--------------------+---------------------|
| TabPFN       |             0.8465 |             0.8434 |             0.8465 |              0.8469 |
| LightGBM     |             0.8328 |             0.8241 |             0.8303 |              0.8323 |
| XGBoost      |             0.7338 |             0.814  |             0.822  |              0.8216 |
| CatBoost     |             0.7534 |             0.8106 |             0.8302 |              0.8374 |
| RandomForest |             0.8326 |             0.826  |             0.832  |              0.8328 |
| Linear       |             0.8445 |             0.8446 |             0.8449 |              0.8455 |
+--------------+--------------------+--------------------+--------------------+---

TabPFN is **flat**: its pretrained prior handles a hole whether or not it ever saw one in this training set. XGBoost and CatBoost **fall off a cliff at 0% train-missing** (roughly minus 0.1 AUC), and even 1% training missingness rescues them, because once a few holes appear in training the model can *learn* which way to send them. That is exactly the production trap that passes cross-validation and then quietly tanks when a feed changes.

RandomForest and LightGBM stay flat on this column, but that is not a ranking of robustness: which model cliffs is not chance, it is each library's default rule for a value missing only at inference, and the next cells make that precise. The only model that never dips, on every setup we tried, is TabPFN.

### Why they cliff: the default direction

Native handling means each model has a rule for *where to send a missing value*. When a column was never missing in training, that rule reduces to a blind default, and the defaults differ by library:

| model | default for a value missing only at inference | routes toward |
|---|---|---|
| **CatBoost** | `nan_mode="Min"` (the default): a NaN is treated as the *minimum*, below every real value | the low end |
| **XGBoost** | the direction is learned from training-missing; with none to learn from, it falls back to a fixed side (the right / high child) | the high end |
| **RandomForest** (sklearn) | the missing sample goes to the child with the *most training samples* | the majority, near the middle |
| **TabPFN** | mean-impute plus indicator: the hole is filled with the feature *mean* | the middle |
| **LightGBM** | `use_missing` / `zero_as_missing` are documented, but the no-training-missing direction is *not*; it is emergent from the trees | data-dependent |

Sources: [scikit-learn](https://scikit-learn.org/stable/modules/tree.html), [CatBoost](https://catboost.ai/docs/en/concepts/algorithm-missing-values-processing), [LightGBM](https://lightgbm.readthedocs.io/en/latest/Advanced-Topics.html), [XGBoost](https://xgboost.readthedocs.io/en/stable/faq.html).

A value routed to an *extreme* is catastrophic when the missingness actually sits at the *opposite* extreme; a value routed to the *middle* only loses the column's information. The next cell makes that visible: a dominant feature `x` is made missing on its **high** tail versus its **low** tail (training always sees it, so each model falls back to its blind default), and we read off where each model sends a hole and what it costs.

In [14]:
# Control WHERE a feature goes missing: a dominant predictor x, made missing on its high tail vs its
# low tail. Training always sees x, so each model falls back to its blind default direction.
def make_vendor_data(n=8000, seed=0):
    rng = np.random.RandomState(seed)
    x = rng.randn(n)
    other = rng.randn(n)
    logit = 2.0 * x + 0.5 * other
    label = (rng.rand(n) < 1 / (1 + np.exp(-logit))).astype(int)
    return pd.DataFrame({"x": x, "other": other}), label

vdata, vlabel = make_vendor_data()
half = len(vdata) // 2
v_train, v_test = vdata.iloc[:half], vdata.iloc[half:].reset_index(drop=True)
vy_train, vy_test = vlabel[:half], vlabel[half:]

def route_fraction(model):
    """Where a missing x lands: 0 = the low-x prediction, 1 = the high-x prediction, ~0.5 = the middle."""
    prediction_at_low_x = model.predict_proba(v_test.assign(x=-2.0))[:, 1].mean()
    prediction_at_high_x = model.predict_proba(v_test.assign(x=2.0))[:, 1].mean()
    prediction_when_missing = model.predict_proba(v_test.assign(x=np.nan))[:, 1].mean()
    return (prediction_when_missing - prediction_at_low_x) / (prediction_at_high_x - prediction_at_low_x)

def missing_tail(frame, side, frac=0.30):
    """Make x missing on its high tail or its low tail (MNAR: the gap depends on x's own value)."""
    frame = frame.copy()
    x_values = frame["x"].values
    threshold = np.quantile(x_values, 1 - frac) if side == "high" else np.quantile(x_values, frac)
    to_hide = x_values >= threshold if side == "high" else x_values <= threshold
    frame.loc[to_hide, "x"] = np.nan
    return frame

summary = {}
for model_name, make_model in tqdm(MODELS.items(), leave=False):
    model = make_model().fit(v_train, vy_train)
    base = roc_auc_score(vy_test, model.predict_proba(v_test)[:, 1])
    drop_high = base - roc_auc_score(vy_test, model.predict_proba(missing_tail(v_test, "high"))[:, 1])
    drop_low = base - roc_auc_score(vy_test, model.predict_proba(missing_tail(v_test, "low"))[:, 1])
    summary[model_name] = {
        "routing_frac": round(route_fraction(model), 2),
        "drop if missing-HIGH x": round(drop_high, 3),
        "drop if missing-LOW x": round(drop_low, 3),
    }

printdf(pd.DataFrame(summary).T, nrows=len(MODELS))

  0%|                                                               | 0/6 [00:00<?, ?it/s]

 17%|█████████▏                                             | 1/6 [00:14<01:13, 14.64s/it]

 33%|██████████████████▎                                    | 2/6 [00:14<00:24,  6.09s/it]

 67%|████████████████████████████████████▋                  | 4/6 [00:16<00:05,  2.76s/it]

 83%|█████████████████████████████████████████████▊         | 5/6 [00:17<00:02,  2.34s/it]

+--------------+----------------+--------------------------+-------------------------+
|              |   routing_frac |   drop if missing-HIGH x |   drop if missing-LOW x |
|--------------+----------------+--------------------------+-------------------------|
| TabPFN       |           0.48 |                    0.072 |                   0.051 |
| LightGBM     |           0.49 |                    0.075 |                   0.07  |
| XGBoost      |           1.01 |                    0.001 |                   0.391 |
| CatBoost     |           0.01 |                    0.41  |                   0.006 |
| RandomForest |           0.45 |                    0.086 |                   0.061 |
| Linear       |           0.49 |                    0.066 |                   0.06  |
+--------------+----------------+--------------------------+-------------------------+


`routing_frac` reads where a missing `x` lands (0 = the low-`x` prediction, 1 = the high-`x` prediction, ~0.5 = the middle), and the two drops are the AUC lost when `x` goes missing on its high or low tail.

The pattern is exactly the documented defaults. **CatBoost** routes low (frac ~0), so it is fine when the hole is at low `x` and collapses when it is at high `x`. **XGBoost** routes high (frac ~1), the mirror image: it dies on low-missing. *Same data, opposite cliffs*, because their fixed defaults point opposite ways, which is why "which library breaks" looks random across datasets but is not. **RandomForest** and **TabPFN** route to the middle, so they only pay the unavoidable information cost in both directions, never a routing inversion. **LightGBM** sits in the middle here too, but it is the one whose direction the docs do not fix, so on a different column it can swing to an extreme and cliff.

The rule of thumb: a model that imputes to the center (TabPFN, by construction) cannot be catastrophically wrong about *direction*; a model that routes to an extreme can, and whether it bites depends on where your data actually goes missing, which is unknowable in advance.

### Mechanism shift: when the reason changes

The harder failure. A column's missingness is informative, but the rule changes between training
and production. We make a noise column `W` whose only signal is *whether* it is missing: in
training "missing" means label 1, and at test the rule flips so "missing" means label 0.

In [15]:
# W is pure noise. Only WHETHER it is missing carries signal, and we flip that rule at test.
rng = np.random.RandomState(0)
X = rng.randn(8000, 8)
W = rng.randn(8000)
label = (rng.rand(8000) < 1 / (1 + np.exp(-1.5 * X[:, 0]))).astype(int)

def build_missingness_shift_dataset(rows, features, noise_column, labels,
                                    p_missing_if_1, p_missing_if_0, seed):
    """Assemble the feature frame for `rows`, making the noise column missing with one
    probability when the label is 1 and another when it is 0. The source arrays are passed
    in (features, noise_column, labels) so the rule is explicit, not read from globals."""
    noise = noise_column[rows].copy()
    missing_probability = np.where(labels[rows] == 1, p_missing_if_1, p_missing_if_0)
    holes = np.random.RandomState(seed).rand(len(noise)) < missing_probability
    noise[holes] = np.nan
    columns = [f"x{i}" for i in range(features.shape[1])] + ["W"]
    return pd.DataFrame(np.column_stack([features[rows], noise]), columns=columns)

train_rows = slice(0, 4000)
test_rows = slice(4000, 8000)
y_train, y_test = label[:4000], label[4000:]

train = build_missingness_shift_dataset(train_rows, X, W, label, p_missing_if_1=0.75, p_missing_if_0=0.25, seed=1)   # missing -> label 1
test_stable = build_missingness_shift_dataset(test_rows, X, W, label, p_missing_if_1=0.75, p_missing_if_0=0.25, seed=2)   # same rule
test_shifted = build_missingness_shift_dataset(test_rows, X, W, label, p_missing_if_1=0.25, p_missing_if_0=0.75, seed=3)  # rule flipped

scores = {}
for model_name, make_model in tqdm(MODELS.items(), leave=False):
    scores[model_name] = {
        "stable": auc(make_model(), train, y_train, test_stable, y_test),
        "shift (rule flips)": auc(make_model(), train, y_train, test_shifted, y_test),
    }

shift_table = pd.DataFrame(scores).T
printdf(shift_table, nrows=len(MODELS))
print("\nSame table, relative to the stable rule (the collapse when the rule flips):")
printdf(relative_to(shift_table, "stable"), nrows=len(MODELS))

  0%|                                                               | 0/6 [00:00<?, ?it/s]

 17%|█████████▏                                             | 1/6 [00:06<00:33,  6.80s/it]

 33%|██████████████████▎                                    | 2/6 [00:06<00:11,  2.90s/it]

 50%|███████████████████████████▌                           | 3/6 [00:07<00:04,  1.66s/it]

 67%|████████████████████████████████████▋                  | 4/6 [00:10<00:04,  2.24s/it]

 83%|█████████████████████████████████████████████▊         | 5/6 [00:14<00:02,  2.93s/it]

100%|███████████████████████████████████████████████████████| 6/6 [00:14<00:00,  2.06s/it]

+--------------+----------+----------------------+
|              |   stable |   shift (rule flips) |
|--------------+----------+----------------------|
| TabPFN       |   0.8822 |               0.5806 |
| LightGBM     |   0.8655 |               0.5725 |
| XGBoost      |   0.8548 |               0.5754 |
| CatBoost     |   0.8733 |               0.5818 |
| RandomForest |   0.8656 |               0.5619 |
| Linear       |   0.8119 |               0.8118 |
+--------------+----------+----------------------+

Same table, relative to the stable rule (the collapse when the rule flips):
+--------------+----------+----------------------+
|              |   stable |   shift (rule flips) |
|--------------+----------+----------------------|
| TabPFN       |        0 |              -0.3016 |
| LightGBM     |        0 |              -0.293  |
| XGBoost      |        0 |              -0.2794 |
| CatBoost     |        0 |              -0.2915 |
| RandomForest |        0 |              -0.3037 |
| Lin

When the rule is stable, every model exploits the informative missingness and gains. When it flips,
they all collapse, well below the score they would get by ignoring `W` entirely, and **TabPFN is
right there with the other native handlers** (having relied on the signal most, it has the most to lose). The one exception is the **linear** model: it imputes `W` and never used the missingness, so it neither gains under the stable rule (0.81, below the others) nor collapses under the flip (0.81, far above them). The model blind to the informative missingness is the only one the shift cannot betray.

This is the trap TabPFN does not escape. Its prior protects you from the *absence* of training
missingness (the vendor trap), not from a *changing* missingness-to-label relationship. That one
you monitor (watch the missing-indicator's distribution drift over time, for example with a population stability index), you do not model your way
out of it.

## Takeaways

- TabPFN does mean-impute-plus-indicator by default, so most of the classical missing-value toolkit
  is redundant for it: native, mean-fill, sentinel, and hand-built indicators all land within noise.
- The one survivor is **model-based imputation of a genuinely complex recoverable column**, where
  TabPFN's internal reconstruction hits a complexity ceiling.
- The **vendor trap** (a column missing only at inference) is where TabPFN clearly wins: flat, while
  the gradient-boosting models cliff.
- **Mechanism shift** is the honest limit: a changing missingness rule fools TabPFN as much as
  anyone, so you monitor it.
- One thread we leave for the **Correlation** chapter: when several columns go missing *together*,
  the missingness pattern is itself a correlated signal. The per-feature indicator channel above puts
  that joint pattern in front of attention for free, while a gradient-boosting model needs you to add the indicators
  by hand. Whether that gap is real, and how deep the co-missingness has to go before it shows up, we
  test there.

The throughline: native handling models the *routing* of missingness learned from training. TabPFN's
prior covers the routing you never saw, but neither covers a routing that changes underneath you.